In [ ]:
import os
from pathlib import Path

# Ensure relative paths resolve from the project root when running from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

In [ ]:
import config

# Model Evaluation Summary

Compute all performance metrics directly from the detailed per-image results CSV.
This notebook is the single source of truth — no separate aggregation pipeline needed.

**Two averaging methods:**
- **Micro-averaging:** Pool all TP/FP/FN, then compute P and R. Each bubble counts equally. High-count images dominate.
- **Macro-averaging:** Compute P and R per image first, then average. Each image counts equally regardless of bubble count.

Outputs:
- Printed summary tables (both averaging methods)
- `performance_results.xlsx` with all sheets

In [8]:
import pandas as pd
import numpy as np

# ── Load data ──
# Adjust path if your CSV is in a different location
CSV_PATH = config.BENCHMARK_OUT_DIR / "metrics" / "detailed_results.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} images")
print(
    f"Scenarios: {df['placement'].nunique()} positions × {df['xanthan'].nunique()} xanthan × {df['rpm'].nunique()} RPM"
)
df.head(3)

Loaded 36 images
Scenarios: 3 positions × 3 xanthan × 2 RPM


,model,split,conf,iou,mask_thr,image,placement,rpm,lmin,xanthan,...,medium_gt,medium_recall,large_tp,large_fn,large_gt,large_recall,xlarge_tp,xlarge_fn,xlarge_gt,xlarge_recall
0,v2,test,0.22,0.35,0.3,0-00-xanthan-150-rpm-90-lmin-place-2-845hpc1z-...,place_2,150,90,0.0,...,37,0.837838,8,3,11,0.727273,0,0,0,NaN
1,v2,test,0.22,0.35,0.3,0-00-xanthan-150-rpm-90-lmin-place-2-845hrt5n-...,place_2,150,90,0.0,...,16,0.937500,3,4,7,0.428571,0,0,0,NaN
2,v2,test,0.22,0.35,0.3,0-00-xanthan-150-rpm-90-lmin-place-4-84faxvtv-...,place_4,150,90,0.0,...,25,0.800000,4,2,6,0.666667,0,0,0,NaN


## Helper Functions

In [9]:
def compute_micro(subset):
    """Micro-averaging: pool TP/FP/FN across all images, then compute P and R.
    Each bubble counts equally — images with more bubbles have more influence."""
    tp = int(subset["tp"].sum())
    fp = int(subset["fp"].sum())
    fn = int(subset["fn"].sum())
    gt = int(subset["gt_count"].sum())
    pred = int(subset["pred_count"].sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "GT": gt,
        "Pred": pred,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "N_images": len(subset),
    }


def compute_macro(subset):
    """Macro-averaging: compute P and R per image, then average.
    Each image counts equally regardless of bubble count."""
    img_p = subset["precision"]
    img_r = subset["recall"]
    return {
        "Precision": round(img_p.mean(), 4),
        "Recall": round(img_r.mean(), 4),
        "Precision_std": round(img_p.std(ddof=1), 4) if len(subset) > 1 else np.nan,
        "Recall_std": round(img_r.std(ddof=1), 4) if len(subset) > 1 else np.nan,
        "N_images": len(subset),
        "GT": int(subset["gt_count"].sum()),
    }


def compute_size_recall(subset):
    """Recall per bubble size class (always micro-averaged — this is bubble-level by nature)."""
    size_classes = ["tiny", "small", "medium", "large", "xlarge"]
    rows = []
    for sc in size_classes:
        tp = int(subset[f"{sc}_tp"].sum())
        fn = int(subset[f"{sc}_fn"].sum())
        gt = tp + fn
        recall = round(tp / gt, 4) if gt > 0 else np.nan
        rows.append({"Size": sc.capitalize(), "TP": tp, "FN": fn, "GT": gt, "Recall": recall})
    result = pd.DataFrame(rows)
    total_gt = result["GT"].sum()
    result["Share (%)"] = (result["GT"] / total_gt * 100).round(1)
    return result


def build_summary(df, groupby_col, label, compute_fn):
    """Build a summary table for a given grouping using the specified averaging method."""
    rows = []
    for name, sub in df.groupby(groupby_col):
        r = compute_fn(sub)
        r[label] = name
        rows.append(r)
    return pd.DataFrame(rows)

## Compute All Summary Tables

Each grouping is computed with both micro- and macro-averaging.

In [10]:
# ── Overall ──
overall_micro = compute_micro(df)
overall_macro = compute_macro(df)

# ── By xanthan ──
by_xanthan_micro = build_summary(df, "xanthan", "Xanthan (wt%)", compute_micro)
by_xanthan_macro = build_summary(df, "xanthan", "Xanthan (wt%)", compute_macro)

# ── By RPM ──
# Add a readable condition label first
df["_condition"] = df.apply(lambda r: f"{int(r.rpm)} min⁻¹, {int(r.lmin)} L min⁻¹", axis=1)
by_rpm_micro = build_summary(df, "_condition", "Condition", compute_micro)
by_rpm_macro = build_summary(df, "_condition", "Condition", compute_macro)

# ── By position ──
by_pos_micro = build_summary(df, "placement", "Position", compute_micro)
by_pos_macro = build_summary(df, "placement", "Position", compute_macro)

# ── Bubble size recall (micro only — this is inherently bubble-level) ──
size_recall = compute_size_recall(df)

# ── All 18 combinations (both methods) ──
combos_micro = []
combos_macro = []
for (placement, xanthan, rpm), sub in df.groupby(["placement", "xanthan", "rpm"]):
    lmin = int(sub["lmin"].iloc[0])
    base = {"Position": placement, "Xanthan (wt%)": xanthan, "RPM": int(rpm), "L/min": lmin}

    r_mi = compute_micro(sub)
    r_mi.update(base)
    combos_micro.append(r_mi)

    r_ma = compute_macro(sub)
    r_ma.update(base)
    combos_macro.append(r_ma)

combos_micro = pd.DataFrame(combos_micro)
combos_macro = pd.DataFrame(combos_macro)

print("All tables computed (micro + macro).")

All tables computed (micro + macro).


## Sanity Checks

In [11]:
checks_passed = True

c1 = (df["tp"] + df["fn"] == df["gt_count"]).all()
print(f"OK TP + FN == GT for all images: {c1}")
checks_passed &= c1

c2 = (df["tp"] + df["fp"] == df["pred_count"]).all()
print(f"OK TP + FP == Pred for all images: {c2}")
checks_passed &= c2

size_cols_gt = ["tiny_gt", "small_gt", "medium_gt", "large_gt", "xlarge_gt"]
size_gt_sum = df[size_cols_gt].fillna(0).sum(axis=1).astype(int)
c3 = (size_gt_sum == df["gt_count"]).all()
print(f"OK Size-class GT sums match total GT: {c3}")
checks_passed &= c3

scenario_counts = df.groupby(["xanthan", "rpm", "placement"]).size()
deviations = scenario_counts[scenario_counts != 2]
c4 = len(deviations) == 0
print(f"OK All scenarios have exactly 2 images: {c4}")
checks_passed &= c4

n_scenarios = df["xanthan"].nunique() * df["rpm"].nunique() * df["placement"].nunique()
print(f"\nTotal: {len(df)} images across {n_scenarios} scenarios")
print(f"All checks passed: {checks_passed}")

OK TP + FN == GT for all images: True
OK TP + FP == Pred for all images: True
OK Size-class GT sums match total GT: True
OK All scenarios have exactly 2 images: True

Total: 36 images across 18 scenarios
All checks passed: True


## Summary Tables: Micro vs Macro

Micro-averaging weights by bubble count (images with more bubbles dominate).
Macro-averaging weights each image equally (better for comparing across conditions with very different bubble counts).

In [12]:
def print_comparison(label, micro_rows, macro_rows, label_col, label_width=28):
    """Print micro vs macro side by side."""
    hdr = f"  {'':>{label_width}s}  {'── Micro ──':>14s}  {'── Macro ──':>14s}     {'Δ Prec':>7s}  {'Δ Recall':>8s}"
    sub_hdr = f"  {'':>{label_width}s}  {'Prec':>6s}  {'Recall':>6s}  {'Prec':>6s}  {'Recall':>6s}"
    print(f"\n{label}")
    print("=" * 90)
    print(hdr)
    print(sub_hdr)
    print("-" * 90)

    for i in range(len(micro_rows)):
        mi = micro_rows.iloc[i] if hasattr(micro_rows, "iloc") else micro_rows[i]
        ma = macro_rows.iloc[i] if hasattr(macro_rows, "iloc") else macro_rows[i]
        name = mi[label_col] if isinstance(mi, dict) else mi[label_col]
        mi_p, mi_r = mi["Precision"], mi["Recall"]
        ma_p, ma_r = ma["Precision"], ma["Recall"]
        dp = ma_p - mi_p
        dr = ma_r - mi_r
        print(
            f"  {str(name):>{label_width}s}  {mi_p:6.4f}  {mi_r:6.4f}  {ma_p:6.4f}  {ma_r:6.4f}     {dp:+7.4f}  {dr:+8.4f}"
        )
    print("=" * 90)


# Overall
print("OVERALL")
print(f"  Micro:  P={overall_micro['Precision']:.4f}  R={overall_micro['Recall']:.4f}")
print(
    f"  Macro:  P={overall_macro['Precision']:.4f}  R={overall_macro['Recall']:.4f}  (±{overall_macro['Precision_std']:.4f} / ±{overall_macro['Recall_std']:.4f})"
)

print_comparison("BY XANTHAN CONCENTRATION", by_xanthan_micro, by_xanthan_macro, "Xanthan (wt%)")

print_comparison("BY OPERATING CONDITION", by_rpm_micro, by_rpm_macro, "Condition")

print_comparison("BY REACTOR POSITION", by_pos_micro, by_pos_macro, "Position")

print("\n\nBubble Size Performance (micro-averaged, bubble-level)")
print("=" * 60)
print(size_recall.to_string(index=False))

OVERALL
  Micro:  P=0.8612  R=0.6677
  Macro:  P=0.8607  R=0.7787  (±0.0652 / ±0.1240)

BY XANTHAN CONCENTRATION
                                   ── Micro ──     ── Macro ──      Δ Prec  Δ Recall
                                  Prec  Recall    Prec  Recall
------------------------------------------------------------------------------------------
                           0.0  0.8521  0.8540  0.8706  0.8668     +0.0185   +0.0128
                         0.125  0.8528  0.7677  0.8584  0.7954     +0.0056   +0.0277
                          0.25  0.8706  0.5794  0.8531  0.6740     -0.0175   +0.0946

BY OPERATING CONDITION
                                   ── Micro ──     ── Macro ──      Δ Prec  Δ Recall
                                  Prec  Recall    Prec  Recall
------------------------------------------------------------------------------------------
         150 min⁻¹, 90 L min⁻¹  0.8582  0.6246  0.8449  0.7045     -0.0133   +0.0799
          75 min⁻¹, 45 L min⁻¹  0.8691  0.818

## All 18 Combinations: Micro vs Macro

In [13]:
# Merge micro and macro for side-by-side view
combo_compare = combos_micro[["Position", "Xanthan (wt%)", "RPM", "GT", "N_images"]].copy()
combo_compare["P_micro"] = combos_micro["Precision"]
combo_compare["R_micro"] = combos_micro["Recall"]

# Align macro by same sort order
combos_macro_sorted = combos_macro.sort_values(["Position", "Xanthan (wt%)", "RPM"]).reset_index(
    drop=True
)
combos_micro_sorted = combos_micro.sort_values(["Position", "Xanthan (wt%)", "RPM"]).reset_index(
    drop=True
)

combo_compare = combos_micro_sorted[["Position", "Xanthan (wt%)", "RPM", "GT"]].copy()
combo_compare["P_micro"] = combos_micro_sorted["Precision"]
combo_compare["R_micro"] = combos_micro_sorted["Recall"]
combo_compare["P_macro"] = combos_macro_sorted["Precision"]
combo_compare["R_macro"] = combos_macro_sorted["Recall"]
combo_compare["ΔP"] = combo_compare["P_macro"] - combo_compare["P_micro"]
combo_compare["ΔR"] = combo_compare["R_macro"] - combo_compare["R_micro"]

print("All 18 Combinations: Micro vs Macro")
print(
    "Large |Δ| values indicate that the two images in a scenario have very different bubble counts."
)
print()
print(combo_compare.to_string(index=False, float_format="%.4f"))

All 18 Combinations: Micro vs Macro
Large |Δ| values indicate that the two images in a scenario have very different bubble counts.

Position  Xanthan (wt%)  RPM   GT  P_micro  R_micro  P_macro  R_macro      ΔP      ΔR
 place_2         0.0000   75   77   0.8780   0.9351   0.8769   0.9333 -0.0011 -0.0018
 place_2         0.0000  150  108   0.7658   0.7870   0.7641   0.7809 -0.0017 -0.0061
 place_2         0.1250   75  114   0.8632   0.8860   0.8603   0.8809 -0.0029 -0.0051
 place_2         0.1250  150 1160   0.8964   0.7388   0.8961   0.7387 -0.0003 -0.0001
 place_2         0.2500   75   93   0.7619   0.8602   0.7582   0.8569 -0.0037 -0.0033
 place_2         0.2500  150 1941   0.9154   0.5131   0.9159   0.5163  0.0005  0.0032
 place_4         0.0000   75   46   0.9773   0.9348   0.9821   0.9361  0.0048  0.0013
 place_4         0.0000  150   81   0.7952   0.8148   0.7922   0.8123 -0.0030 -0.0025
 place_4         0.1250   75  361   0.9080   0.8199   0.9069   0.8199 -0.0011  0.0000
 place_4

## Export to Excel

Generates `performance_results.xlsx` with both micro and macro results.

In [14]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

XLSX_PATH = "performance_results.xlsx"
FONT_NAME = "Arial"

# ── Styles ──
header_font = Font(name=FONT_NAME, bold=True, size=11, color="FFFFFF")
header_fill = PatternFill("solid", fgColor="1E293B")
section_font = Font(name=FONT_NAME, bold=True, size=10, color="334155")
section_fill = PatternFill("solid", fgColor="F1F5F9")
data_font = Font(name=FONT_NAME, size=10)
bold_font = Font(name=FONT_NAME, size=10, bold=True)
overall_fill = PatternFill("solid", fgColor="EFF6FF")
micro_fill = PatternFill("solid", fgColor="DBEAFE")
macro_fill = PatternFill("solid", fgColor="FEF3C7")
thin_border = Border(
    left=Side(style="thin", color="CCCCCC"),
    right=Side(style="thin", color="CCCCCC"),
    top=Side(style="thin", color="CCCCCC"),
    bottom=Side(style="thin", color="CCCCCC"),
)
center = Alignment(horizontal="center", vertical="center")
left_align = Alignment(horizontal="left", vertical="center")


def style_header(ws, row, cols):
    for c in range(1, cols + 1):
        cell = ws.cell(row=row, column=c)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = center
        cell.border = thin_border


def style_row(ws, row, cols, font=data_font, fill=None):
    for c in range(1, cols + 1):
        cell = ws.cell(row=row, column=c)
        cell.font = font
        cell.border = thin_border
        cell.alignment = center if c >= 2 else left_align
        if fill:
            cell.fill = fill


def write_section(ws, row, text, cols):
    ws.cell(row=row, column=1, value=text).font = section_font
    for c in range(1, cols + 1):
        ws.cell(row=row, column=c).fill = section_fill
        ws.cell(row=row, column=c).border = thin_border


def set_fmt(ws, r1, r2, c1, c2, fmt):
    for row_cells in ws.iter_rows(min_row=r1, max_row=r2, min_col=c1, max_col=c2):
        for cell in row_cells:
            if isinstance(cell.value, float):
                cell.number_format = fmt


wb = Workbook()

# ═══════════════════════════════════════════
# Sheet 1: Summary (Micro + Macro side by side)
# ═══════════════════════════════════════════
ws1 = wb.active
ws1.title = "Summary"
COLS1 = 6

r = 1
for c, h in enumerate(["", "P (micro)", "R (micro)", "P (macro)", "R (macro)", "N images"], 1):
    ws1.cell(row=r, column=c, value=h)
style_header(ws1, r, COLS1)
# Color sub-headers
for c in [2, 3]:
    ws1.cell(row=r, column=c).fill = PatternFill("solid", fgColor="1E40AF")
for c in [4, 5]:
    ws1.cell(row=r, column=c).fill = PatternFill("solid", fgColor="92400E")

r = 2
ws1.cell(
    row=r,
    column=1,
    value=f"Overall ({overall_micro['N_images']} images, {overall_micro['GT']} GT bubbles)",
)
ws1.cell(row=r, column=2, value=overall_micro["Precision"])
ws1.cell(row=r, column=3, value=overall_micro["Recall"])
ws1.cell(row=r, column=4, value=overall_macro["Precision"])
ws1.cell(row=r, column=5, value=overall_macro["Recall"])
ws1.cell(row=r, column=6, value=overall_micro["N_images"])
style_row(ws1, r, COLS1, font=bold_font, fill=overall_fill)


def write_group(ws, start_row, section_label, micro_df, macro_df, label_col):
    r = start_row
    write_section(ws, r, section_label, COLS1)
    for i in range(len(micro_df)):
        r += 1
        mi = micro_df.iloc[i]
        ma = macro_df.iloc[i]
        ws.cell(row=r, column=1, value=str(mi[label_col]))
        ws.cell(row=r, column=2, value=mi["Precision"])
        ws.cell(row=r, column=3, value=mi["Recall"])
        ws.cell(row=r, column=4, value=ma["Precision"])
        ws.cell(row=r, column=5, value=ma["Recall"])
        ws.cell(row=r, column=6, value=mi["N_images"])
        style_row(ws, r, COLS1)
    return r


r = write_group(
    ws1, 3, "Xanthan concentration", by_xanthan_micro, by_xanthan_macro, "Xanthan (wt%)"
)
r = write_group(ws1, r + 1, "Operating condition", by_rpm_micro, by_rpm_macro, "Condition")
r = write_group(ws1, r + 1, "Reactor position", by_pos_micro, by_pos_macro, "Position")

ws1.column_dimensions["A"].width = 44
for col in ["B", "C", "D", "E"]:
    ws1.column_dimensions[col].width = 14
ws1.column_dimensions["F"].width = 12
set_fmt(ws1, 2, r, 2, 5, "0.0000")

# Light background tint for micro vs macro columns
for row_cells in ws1.iter_rows(min_row=2, max_row=r, min_col=2, max_col=3):
    for cell in row_cells:
        if cell.fill == PatternFill() or cell.fill.fgColor.rgb == "00000000":
            cell.fill = micro_fill
for row_cells in ws1.iter_rows(min_row=2, max_row=r, min_col=4, max_col=5):
    for cell in row_cells:
        if cell.fill == PatternFill() or cell.fill.fgColor.rgb == "00000000":
            cell.fill = macro_fill

# ═══════════════════════════════════════════
# Sheet 2: Bubble Size Performance
# ═══════════════════════════════════════════
ws2 = wb.create_sheet("Bubble Size Performance")
COLS2 = 6

r = 1
for c, h in enumerate(["Size Class", "Recall", "TP", "FN", "GT", "Share (%)"], 1):
    ws2.cell(row=r, column=c, value=h)
style_header(ws2, r, COLS2)

for _, rd in size_recall.iterrows():
    r += 1
    ws2.cell(row=r, column=1, value=rd["Size"])
    ws2.cell(row=r, column=2, value=rd["Recall"])
    ws2.cell(row=r, column=3, value=rd["TP"])
    ws2.cell(row=r, column=4, value=rd["FN"])
    ws2.cell(row=r, column=5, value=rd["GT"])
    ws2.cell(row=r, column=6, value=rd["Share (%)"])
    style_row(ws2, r, COLS2)

r += 1
total_tp = int(size_recall["TP"].sum())
total_gt = int(size_recall["GT"].sum())
ws2.cell(row=r, column=1, value="Total")
ws2.cell(row=r, column=2, value=round(total_tp / total_gt, 4))
ws2.cell(row=r, column=3, value=total_tp)
ws2.cell(row=r, column=4, value=int(size_recall["FN"].sum()))
ws2.cell(row=r, column=5, value=total_gt)
ws2.cell(row=r, column=6, value=100.0)
style_row(ws2, r, COLS2, font=bold_font, fill=overall_fill)

ws2.column_dimensions["A"].width = 16
for col in ["B", "C", "D", "E", "F"]:
    ws2.column_dimensions[col].width = 12
set_fmt(ws2, 2, r, 2, 2, "0.0000")
set_fmt(ws2, 2, r, 6, 6, "0.0")

# ═══════════════════════════════════════════
# Sheet 3: All 18 Combinations (micro + macro)
# ═══════════════════════════════════════════
ws3 = wb.create_sheet("All Combinations")
COLS3 = 12

r = 1
headers3 = [
    "Position",
    "Xanthan (wt%)",
    "RPM",
    "L/min",
    "P (micro)",
    "R (micro)",
    "P (macro)",
    "R (macro)",
    "TP",
    "FP",
    "FN",
    "GT",
]
for c, h in enumerate(headers3, 1):
    ws3.cell(row=r, column=c, value=h)
style_header(ws3, r, COLS3)
for c in [5, 6]:
    ws3.cell(row=r, column=c).fill = PatternFill("solid", fgColor="1E40AF")
for c in [7, 8]:
    ws3.cell(row=r, column=c).fill = PatternFill("solid", fgColor="92400E")

combos_mi_s = combos_micro.sort_values(["Position", "Xanthan (wt%)", "RPM"]).reset_index(drop=True)
combos_ma_s = combos_macro.sort_values(["Position", "Xanthan (wt%)", "RPM"]).reset_index(drop=True)

prev_pos = None
for i in range(len(combos_mi_s)):
    mi = combos_mi_s.iloc[i]
    ma = combos_ma_s.iloc[i]
    if mi["Position"] != prev_pos:
        if prev_pos is not None:
            r += 1
        prev_pos = mi["Position"]
    r += 1
    ws3.cell(row=r, column=1, value=mi["Position"])
    ws3.cell(row=r, column=2, value=mi["Xanthan (wt%)"])
    ws3.cell(row=r, column=3, value=mi["RPM"])
    ws3.cell(row=r, column=4, value=mi["L/min"])
    ws3.cell(row=r, column=5, value=mi["Precision"])
    ws3.cell(row=r, column=6, value=mi["Recall"])
    ws3.cell(row=r, column=7, value=ma["Precision"])
    ws3.cell(row=r, column=8, value=ma["Recall"])
    ws3.cell(row=r, column=9, value=mi["TP"])
    ws3.cell(row=r, column=10, value=mi["FP"])
    ws3.cell(row=r, column=11, value=mi["FN"])
    ws3.cell(row=r, column=12, value=mi["GT"])
    style_row(ws3, r, COLS3)

ws3.column_dimensions["A"].width = 14
ws3.column_dimensions["B"].width = 16
ws3.column_dimensions["C"].width = 10
ws3.column_dimensions["D"].width = 10
for col in ["E", "F", "G", "H"]:
    ws3.column_dimensions[col].width = 14
for col in ["I", "J", "K", "L"]:
    ws3.column_dimensions[col].width = 10
set_fmt(ws3, 2, r, 5, 8, "0.0000")

# Tint micro/macro columns
for row_cells in ws3.iter_rows(min_row=2, max_row=r, min_col=5, max_col=6):
    for cell in row_cells:
        cell.fill = micro_fill
for row_cells in ws3.iter_rows(min_row=2, max_row=r, min_col=7, max_col=8):
    for cell in row_cells:
        cell.fill = macro_fill

# ═══════════════════════════════════════════
# Sheet 4: Per-Image Detail
# ═══════════════════════════════════════════
ws4 = wb.create_sheet("Per-Image Detail")

detail_cols = [
    "placement",
    "xanthan",
    "rpm",
    "lmin",
    "image",
    "tp",
    "fp",
    "fn",
    "gt_count",
    "pred_count",
    "precision",
    "recall",
]
df_detail = df[detail_cols].sort_values(["placement", "xanthan", "rpm", "image"])

r = 1
nice_headers = [
    "Position",
    "Xanthan",
    "RPM",
    "L/min",
    "Image",
    "TP",
    "FP",
    "FN",
    "GT",
    "Pred",
    "Precision",
    "Recall",
]
COLS4 = len(nice_headers)
for c, h in enumerate(nice_headers, 1):
    ws4.cell(row=r, column=c, value=h)
style_header(ws4, r, COLS4)

for _, rd in df_detail.iterrows():
    r += 1
    for c, col in enumerate(detail_cols, 1):
        ws4.cell(row=r, column=c, value=rd[col])
        ws4.cell(row=r, column=c).font = Font(name=FONT_NAME, size=9)
        ws4.cell(row=r, column=c).border = thin_border
        if c >= 6:
            ws4.cell(row=r, column=c).alignment = center

ws4.column_dimensions["A"].width = 12
ws4.column_dimensions["B"].width = 10
ws4.column_dimensions["C"].width = 8
ws4.column_dimensions["D"].width = 8
ws4.column_dimensions["E"].width = 50
for col in ["F", "G", "H", "I", "J"]:
    ws4.column_dimensions[col].width = 8
ws4.column_dimensions["K"].width = 12
ws4.column_dimensions["L"].width = 12
set_fmt(ws4, 2, r, 11, 12, "0.0000")

# ── Save ──
wb.save(XLSX_PATH)
print(f"Saved: {XLSX_PATH}")
print(f"Sheets: {wb.sheetnames}")

Saved: performance_results.xlsx
Sheets: ['Summary', 'Bubble Size Performance', 'All Combinations', 'Per-Image Detail']
